# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(metadata['name'])
print(metadata['description'])
print("Published:", metadata.get('datePublished'))
print("License:", metadata.get('license'))
print("Fields (keywords):", metadata.get('keywords'))

## 2. Data Overview
Review available record sets, fields, and their IDs.

#### Note
In the Croissant schema, entities like record sets and fields are referenced by their `@id` fields. We'll list available record sets and the fields/columns they contain.


In [ ]:
# List all record sets and their details
record_sets = dataset.metadata.record_sets
record_set_ids = [rs['@id'] for rs in record_sets]
print("Record sets in dataset:")
for rs in record_sets:
    print(f"  @id: {rs['@id']} | name: {rs.get('name', 'N/A')}")
    print("    Fields/Columns:")
    for field in rs.get('fields', []):
        print(f"      - @id: {field['@id']} | name: {field.get('name', 'N/A')}")

# If available, preview a few records from one record set
record_set_preview_id = record_set_ids[0] if record_set_ids else None
if record_set_preview_id:
    print("\nExample records from record set:", record_set_preview_id)
    for x in dataset.records(record_set=record_set_preview_id):
        print(json.dumps(x, indent=2))
        break  # Show only one example record
else:
    print("No record sets found in schema.")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

This example extracts data from all record sets found in the schema, referencing each by its unique `@id`.


In [ ]:
# Extract data from each record set
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Record set @id: {record_set_id}")
        print("Columns:", df.columns.tolist())
        print(df.head(2))

# For demonstration, select first record set
selected_record_set_id = record_set_ids[0] if record_set_ids else None
if selected_record_set_id:
    print("\nSelected DataFrame columns:", dataframes[selected_record_set_id].columns.tolist())
    dataframes[selected_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalizing numeric fields, and grouping. We reference columns by their `@id` (or field names where `@id` is not a column key).

- Remove outliers
- Normalize numeric fields
- Group by key columns

Use your chosen record set and field/column identifiers from prior steps.

In [ ]:
# EDA: Example numeric and grouping fields
df = dataframes[selected_record_set_id]

# Try to find numeric field by checking for int/float dtype
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]  # List of numeric columns
if numeric_fields:
    numeric_field = numeric_fields[0]
    print(f"Numeric field selected (@id or name): {numeric_field}")
else:
    numeric_field = None

threshold = 10
if numeric_field:
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field
    group_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field]
    if group_fields:
        group_field = group_fields[0]
        print(f"Grouped data by {group_field}: (Mean values)")
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(grouped_df.head())
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset.

Below is an example using matplotlib and seaborn, plotting histograms and boxplots of available numeric and group fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if group_fields:
        plt.figure(figsize=(7,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
This notebook demonstrated a complete workflow for loading, exploring, and analyzing a dataset defined by a Croissant schema using the `mlcroissant` library, referencing entities explicitly by their `@id` fields for reproducibility.

- Loaded metadata and recordsets from FAIR^2 dataset
- Inspected available recordset and field IDs
- Extracted data into DataFrames
- Applied common EDA and visualizations
- All references are made via `@id` where available, ensuring semantic consistency

Further steps could include model development, advanced visualization, or integrating additional metadata from the schema.